In [3]:
from pathlib import Path
import re

import pandas as pd

def resolve_physic_csv() -> Path:
    candidates = [Path.cwd() / "data" / "physic.csv", Path.cwd() / "physic.csv"]
    candidates += [parent / "data" / "physic.csv" for parent in Path.cwd().parents]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError("physic.csv not found from current working directory")

csv_path = resolve_physic_csv()
df = pd.read_csv(csv_path)

text_cols = df.select_dtypes(include=["object", "string"]).columns
text_data = df[text_cols].fillna("").astype(str)

_SUPER_TO_ASCII = {
    "⁰": "0",
    "¹": "1",
    "²": "2",
    "³": "3",
    "⁴": "4",
    "⁵": "5",
    "⁶": "6",
    "⁷": "7",
    "⁸": "8",
    "⁹": "9",
    "⁺": "+",
    "⁻": "-",
}

def collect_unique_chars(frame: pd.DataFrame) -> str:
    chars = set()
    for col in frame.columns:
        for value in frame[col]:
            chars.update(value)
    return "".join(sorted(chars))

def replace_superscripts(text: str) -> str:
    def repl(match: re.Match) -> str:
        base = match.group(1)
        exp = "".join(_SUPER_TO_ASCII[ch] for ch in match.group(2))
        return f"{base}^{exp}"
    return re.sub(r"(\d)([⁰¹²³⁴⁵⁶⁷⁸⁹⁺⁻]+)", repl, text)

def normalize_scientific(text: str) -> str:
    text = replace_superscripts(text)
    text = re.sub(r"\s*[×·]\s*", " * ", text)
    text = re.sub(r"(\d)\s*\.\s*(10\^)", r"\1 * \2", text)
    text = re.sub(r"(\d)\s*\*\s*(10\^)", r"\1 * \2", text)
    text = re.sub(r"10\^([+-]?\d+)", r"10^{\1}", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text

unique_chars_before = collect_unique_chars(text_data)

df_norm = df.copy()
for col in text_cols:
    df_norm[col] = df_norm[col].fillna("").astype(str).map(normalize_scientific)

unique_chars_after = collect_unique_chars(df_norm[text_cols])

diff_mask = df_norm[text_cols] != text_data
changed_rows = diff_mask.any(axis=1).sum()

print(f"Unique chars before: {len(unique_chars_before)}")
print(unique_chars_before)
print(f"Unique chars after: {len(unique_chars_after)}")
print(unique_chars_after)
print(f"Rows with changes: {changed_rows}")

if changed_rows:
    display(
        pd.concat(
            {
                "before": df.loc[diff_mask.any(axis=1), text_cols].head(5),
                "after": df_norm.loc[diff_mask.any(axis=1), text_cols].head(5),
            },
            axis=0,
        )
    )

Unique chars before: 191

 !"$%'()*+,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}°±²³µ·¹½×àáâíô÷ĐđƠơư̂ΔΠΦΨΩαβγδεθκλμπρσφωᵦạảầậặếềểệọỏốổộớờợủứừ–—’′⁰⁴⁵⁶⁷⁸⁹⁻₀₁₂₃₄₅ₑₙₜ⃗ℓΩ⅓⅔→⇒−√∝≈≠≤≥⊥⋅⟂
Unique chars after: 182

 !"$%'()*+,-./0123456789:;<=>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}°±²³µ¹½àáâíô÷ĐđƠơư̂ΔΠΦΨΩαβγδεθκλμπρσφωᵦạảầậặếềểệọỏốổộớờợủứừ–—’′⁻₀₁₂₃₄₅ₑₙₜ⃗ℓΩ⅓⅔→⇒−√∝≈≠≤≥⊥⋅⟂
Rows with changes: 1020


id                                           question  \
before 0  TD401  Calculate the energy stored in capacitor C whe...   
       1  TD402  Calculate the capacitance C of the capacitor, ...   
       2  LD001  Two charges, q1 = 6 × 10^-8 C and q2 = -6 × 10...   
       3  LD002  Three electric charges are placed at three fix...   
       4  LD003  Points A and B are separated by 20 cm in air. ...   
after  0  TD401  Calculate the energy stored in capacitor C whe...   
       1  TD402  Calculate the capacitance C of the capacitor, ...   
       2  LD001  Two charges, q1 = 6 * 10^{-8} C and q2 = -6 * ...   
       3  LD002  Three electric charges are placed at three fix...   
       4  LD003  Points A and B are separated by 20 cm in air. ...   

                                                        cot           answer  \
before 0  Step 1: Identify the given values for capacita...               45   
       1  Step 1: Identify the given values from the que...              100   
       2  Step 1: Identify the given charges and distanc...             0.05   
       3  Step 1: Identify the charges and distances giv...    24.45 × 10^-3   
       4  Step 1: Identify the given charges and distanc...             6.76   
after  0  Step 1: Identify the given values for capacita...               45   
       1  Step 1: Identify the given values from the que...              100   
       2  Step 1: Identify the given charges and distanc...             0.05   
       3  Step 1: Identify the charges and distances giv...  24.45 * 10^{-3}   
       4  Step 1: Identify the given charges and distanc...             6.76   

         unit  
before 0    J  
       1   μF  
       2    N  
       3    N  
       4    N  
after  0    J  
       1   μF  
       2    N  
       3    N  
       4    N